### HW7_1: Scientific Data Preparation and EDA
 Tiago Garcia Guimarães | DACS-630 | Dataset: Child Health and Development Studies (CHDS), babies.csv

### 1. Hypothesis

Maternal smoking during pregnancy is associated with lower infant birth weight.

### 2. Setup and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
babies = pd.read_csv('babies.csv')
babies.head()

,case,bwt,gestation,parity,age,height,weight,smoke
0,1,120,284.0,0,27.0,62.0,100.0,0.0
1,2,113,282.0,0,33.0,64.0,135.0,0.0
2,3,128,279.0,0,28.0,64.0,115.0,1.0
3,4,123,NaN,0,36.0,69.0,190.0,0.0
4,5,108,282.0,0,23.0,67.0,125.0,1.0


### 3. Data Inspection

In [3]:
babies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1236 entries, 0 to 1235
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   case       1236 non-null   int64  
 1   bwt        1236 non-null   int64  
 2   gestation  1223 non-null   float64
 3   parity     1236 non-null   int64  
 4   age        1234 non-null   float64
 5   height     1214 non-null   float64
 6   weight     1200 non-null   float64
 7   smoke      1226 non-null   float64
dtypes: float64(5), int64(3)
memory usage: 77.4 KB


In [4]:
babies.describe()

,case,bwt,gestation,parity,age,height,weight,smoke
count,1236.000000,1236.000000,1223.000000,1236.000000,1234.000000,1214.000000,1200.000000,1226.000000
mean,618.500000,119.576861,279.338512,0.254854,27.255267,64.047776,128.625833,0.394780
std,356.946775,18.236452,16.027693,0.435956,5.781405,2.533409,20.971862,0.489003
min,1.000000,55.000000,148.000000,0.000000,15.000000,53.000000,87.000000,0.000000
25%,309.750000,108.750000,272.000000,0.000000,23.000000,62.000000,114.750000,0.000000
50%,618.500000,120.000000,280.000000,0.000000,26.000000,64.000000,125.000000,0.000000
75%,927.250000,131.000000,288.000000,1.000000,31.000000,66.000000,139.000000,1.000000
max,1236.000000,176.000000,353.000000,1.000000,45.000000,72.000000,250.000000,1.000000


In [5]:
babies.isna() .sum()

case          0
bwt           0
gestation    13
parity        0
age           2
height       22
weight       36
smoke        10
dtype: int64

In [6]:
babies.duplicated() .sum()

np.int64(0)

### 4. Data Cleaning
### 4.1 Missing Data

In [7]:
clean = babies.copy()
clean = clean.dropna(subset=['smoke', 'gestation'])
clean.shape

(1213, 8)

### 4.2 Outliers

In [8]:
clean[clean['gestation'] > 315]

,case,bwt,gestation,parity,age,height,weight,smoke
10,11,140,351.0,0,27.0,68.0,120.0,0.0
152,153,127,336.0,0,29.0,NaN,NaN,0.0
155,156,123,318.0,0,21.0,64.0,152.0,0.0
198,199,123,320.0,0,22.0,66.0,117.0,0.0
234,235,93,318.0,0,31.0,66.0,135.0,0.0
240,241,144,329.0,0,22.0,65.0,190.0,1.0
253,254,117,328.0,0,29.0,65.0,125.0,1.0
279,280,141,319.0,1,20.0,67.0,140.0,1.0
460,461,123,323.0,1,17.0,64.0,140.0,0.0
685,686,115,316.0,1,29.0,64.0,110.0,0.0


In [9]:
removidos = (clean['gestation'] > 315).sum()
clean = clean[clean['gestation'] <= 315]
print('Linhas removidas:', removidos)
print('Linhas restantes:', clean.shape[0])

Linhas removidas: 21
Linhas restantes: 1192


In [10]:
clean = clean.drop([260, 869])
print('Linhas restantes:', clean.shape[0])

Linhas restantes: 1190


Two records (index 260 and 869) were also removed. They showed gestations of about 21 and 26 weeks with birth weights above 3,000 g, a combination that is biologically implausible and most likely a date-recording error. These cases were identified during the EDA (Section 5.3).

### 4.3 Type Conversions and Transformations

In [11]:
clean['smoke'] = clean['smoke'].astype(int)
clean['bwt_grams'] = clean['bwt'] * 28.35
clean['gestation_weeks'] = clean['gestation'] / 7
clean['smoke_label'] = clean['smoke'].map({0: 'Non-smoker', 1: 'Smoker'})
clean['preterm'] = clean['gestation'] < 259
clean.head()

,case,bwt,gestation,parity,age,height,weight,smoke,bwt_grams,gestation_weeks,smoke_label,preterm
0,1,120,284.0,0,27.0,62.0,100.0,0,3402.00,40.571429,Non-smoker,False
1,2,113,282.0,0,33.0,64.0,135.0,0,3203.55,40.285714,Non-smoker,False
2,3,128,279.0,0,28.0,64.0,115.0,1,3628.80,39.857143,Smoker,False
4,5,108,282.0,0,23.0,67.0,125.0,1,3061.80,40.285714,Smoker,False
5,6,136,286.0,0,25.0,62.0,93.0,0,3855.60,40.857143,Non-smoker,False


### 5. Exploratory Data Analysis (EDA)
### 5.1 Birth Weight by Smoking Status
Comparing birth weight between babies of smoking and non-smoking mothers.

In [ ]:
clean.groupby('smoke_label')['bwt_grams'].describe()

In [ ]:
non_smoker = clean[clean['smoke_label'] == 'Non-smoker']['bwt_grams']
smoker = clean[clean['smoke_label'] == 'Smoker']['bwt_grams']

plt.figure(figsize=(7, 5))
plt.boxplot([non_smoker, smoker], flierprops={'markerfacecolor': 'black'})
plt.xticks([1, 2], ['Non-smoker', 'Smoker'])
plt.axhline(2500, color='gray', linestyle='--')
plt.title('Birth Weight by Maternal Smoking Status')
plt.xlabel('Maternal smoking during pregnancy')
plt.ylabel('Birth weight (grams)')
plt.savefig('fig1_bwt_by_smoking.png', dpi=300, bbox_inches='tight')
plt.show()

Note: I tested different versions of this chart and chose the simplest one. A basic boxplot keeps the code easy to read and reproduce, and it still shows the main point: the median, the spread, and the outliers of each group.

### 5.2 Stratified View: Preterm vs Full Term
Checking whether the difference holds within babies born at the same gestational stage.

In [ ]:
clean.groupby(['preterm', 'smoke_label'])['bwt_grams'].describe()

In [ ]:
full_non = clean[(clean['preterm'] == False) & (clean['smoke_label'] == 'Non-smoker')]['bwt_grams']
full_smk = clean[(clean['preterm'] == False) & (clean['smoke_label'] == 'Smoker')]['bwt_grams']
pre_non = clean[(clean['preterm'] == True) & (clean['smoke_label'] == 'Non-smoker')]['bwt_grams']
pre_smk = clean[(clean['preterm'] == True) & (clean['smoke_label'] == 'Smoker')]['bwt_grams']

plt.figure(figsize=(8, 5))
plt.boxplot([full_non, full_smk, pre_non, pre_smk], flierprops={'markerfacecolor': 'black'})
plt.xticks([1, 2, 3, 4], ['Full term\nNon-smoker', 'Full term\nSmoker', 'Preterm\nNon-smoker', 'Preterm\nSmoker'])
plt.axhline(2500, color='gray', linestyle='--')
plt.title('Birth Weight by Smoking Status, Stratified by Gestation')
plt.ylabel('Birth weight (grams)')
plt.savefig('fig2_bwt_stratified.png', dpi=300, bbox_inches='tight')
plt.show()

#### Interpretation
If smoking only shortened pregnancies, the gap between smokers and non-smokers should disappear once babies are compared at the same gestational stage. It does not. Among full-term babies, those born to smoking mothers weigh about 244 g less, and among preterm babies the gap grows to about 394 g. In other words, smoking seems to affect fetal growth itself, not only how long the pregnancy lasts. The preterm smokers' group is the most concerning: its median sits just above the 2,500 g line, so a large share of these babies were born with low birth weight. Still, this group is small (41 babies, against 54 preterm non-smokers), so these numbers should be read as a signal, not as a final answer.

### 5.3 Gestation Length vs Birth Weight
Looking at how birth weight changes with gestation length for each group.

In [ ]:
non = clean[clean['smoke_label'] == 'Non-smoker']
smk = clean[clean['smoke_label'] == 'Smoker']

plt.figure(figsize=(8, 5))
plt.scatter(non['gestation_weeks'], non['bwt_grams'], alpha=0.5, label='Non-smoker')
plt.scatter(smk['gestation_weeks'], smk['bwt_grams'], alpha=0.5, label='Smoker')
plt.axvline(37, color='gray', linestyle='--')
plt.title('Gestation Length vs Birth Weight')
plt.xlabel('Gestation (weeks)')
plt.ylabel('Birth weight (grams)')
plt.legend()
plt.savefig('fig3_gestation_vs_bwt.png', dpi=300, bbox_inches='tight')
plt.show()

#### Interpretation
Birth weight increases as gestation gets longer, which confirms gestation as an important confounder. Even so, at the same gestational week, babies of smoking mothers tend to sit lower on the chart, while the top of the chart is mostly non-smokers. Two records with implausible gestation and weight combinations were found during this step and removed in Section 4.2.

### 6. Confounders
Checking which variables are related to both smoking and birth weight.

In [ ]:
clean.groupby('smoke_label')[['gestation_weeks', 'age', 'height', 'weight', 'parity']].mean()

In [ ]:
clean[['bwt_grams', 'gestation_weeks', 'age', 'height', 'weight', 'parity']].corr()['bwt_grams']

#### Interpretation
A confounder is a third factor that could explain the result instead of smoking. For example, if smokers were much younger or lighter, those differences alone could explain why their babies weigh less.

The tables show that the two groups are very similar:
- Smokers' pregnancies were only about 2 days shorter.
- Smokers were slightly younger (26.8 vs 27.6 years) and slightly lighter (126.7 vs 129.4 lb).
- Height and number of previous children were almost the same.

Gestation length has the strongest link with birth weight: longer pregnancies mean heavier babies. Maternal height and weight show a weaker link, and age almost none.

Since the groups are so similar, these factors do not explain the gap of about 260 g. This matches what the CDC reports: smoking during pregnancy is linked to low birth weight (CDC, 2025). Still, some factors are not in the dataset, such as diet, income and prenatal care, so they cannot be ruled out.

### 7. Export Clean Dataset
Saving the final clean dataset (1,190 rows) for reproducibility.

In [ ]:
clean.to_csv('HW7_1_clean_scientific.csv', index=False)
print('Arquivo salvo com', clean.shape[0], 'linhas e', clean.shape[1], 'colunas')

### 8. Methods & Limitations Memo

### Purpose
This analysis tests one question: is smoking during pregnancy linked to lower birth weight? The data come from the Child Health and Development Studies (CHDS). The study followed pregnancies in the San Francisco Bay Area in the 1960s. The raw file had 1,236 births and 8 columns.

### Missing data
- Smoking (10 missing): this is the main variable of the study. Filling it with a guess would not be honest, so these rows were removed.
- Gestation (13 missing): this is the most important confounder. It is also needed to split the babies into preterm and full term, so these rows were removed too.
- Age, height and weight (2, 22 and 36 missing): these rows were kept. These columns are only used to check confounders, and pandas skips missing values in those calculations.

In total, 23 rows were removed, leaving 1,213.

### Outliers
The rule was simple: remove a value only if it is impossible in real life, not just because it is far from the average.
- 21 records had gestation above 315 days (45 weeks). This is very unlikely and probably a date error, so they were removed.
- 2 records showed babies born at about 21 and 26 weeks weighing more than 3,000 g. That is the weight of a full-term baby, so these were also treated as errors. They were found during the EDA.
- Very low and very high birth weights were kept. They are real, and they are exactly the cases this study cares about.

Final dataset: 1,190 births.

### Transformations
- Birth weight changed from ounces to grams, and gestation from days to weeks. These are the units used in health research.
- Smoking became a 0/1 number, plus a text label (Smoker / Non-smoker) for tables and charts.
- A new preterm column marks babies born before 37 weeks (259 days), the standard limit for preterm birth.
- No log, scaling or winsorizing was used. Birth weight is already fairly balanced, the analysis only compares group averages, and winsorizing would change real values that matter.

### Visuals
- Figure 1: birth weight by smoking status.
- Figure 2: the same comparison, split into preterm and full-term babies.
- Figure 3: gestation length against birth weight.

### Limitations
1. Old and local data: one region, one health plan, in the 1960s. A lot has changed since then, so the results may not apply to today.
2. Smoking is only yes or no: the data do not show how much the mother smoked. It was also self-reported, so some mothers may have said less than the truth.
3. Missing factors: diet, income, alcohol and prenatal care are not in the data. They could explain part of the gap.
4. Small preterm groups: only 54 and 41 babies, so those results are less reliable.
5. Link, not proof: this is observational data. It shows a clear and consistent link, but it cannot prove that smoking alone caused the lower weight.

### 9. Ethics Disclosure

### Sensitive attribute
The main variable, maternal smoking, is a personal behavior. It can also work as a proxy for other factors, such as income, education or stress. A careless reading of the results could blame mothers, when smoking is often linked to life conditions they do not fully control.

### How the results should be used
- The findings describe groups, not individuals. A smoking mother can have a healthy, heavy baby, and a non-smoking mother can have a small one.
- The goal is to support mothers, for example through prenatal care and programs to quit smoking, not to judge them.
- The results show a link, not proof of cause, and should not be presented as stronger than they are.

### Privacy and consent
- The dataset is public and has no names or personal identifiers. Each birth appears only as a case number.
- The data were collected in the 1960s, when consent rules were less strict than today. The participants could not have imagined their data being used in online courses decades later. This is a reason to use the data with respect and only for learning purposes.
- Only the columns needed for the question were used, and no attempt was made to identify any person.

### Representation
The sample comes from one health plan in one region. Groups that did not have access to this health plan are not represented, so the results should not be generalized to every population.

### References

Centers for Disease Control and Prevention. (2025, January 31). *Health effects of cigarettes: Reproductive health*. https://www.cdc.gov/tobacco/about/cigarettes-and-reproductive-health.html

Molin, S. (2021). *Hands-on data analysis with pandas* (2nd ed.). Packt Publishing.

OpenIntro. (n.d.). *Babies* [Data set]. Original data from the Child Health and Development Studies (CHDS). https://www.openintro.org/data/index.php?data=babies